# Results 1 — Panoramic view across 100,526 complex trait GWAS

Every number claimed in this Results subsection, and the per-year discovery curves behind
Figure 1b–d, Extended Data Figures 3 and 10.

Numbers are collected in `numbers` and written to `results/panoramic.json`, which
`tools/check_numbers.py` compares against the manuscript.

In [ ]:
import pandas as pd
from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from pyspark.sql import functions as f

from manuscript_methods import discovery, paper

session = Session(extended_spark_conf={"spark.driver.memory": "40G"})
numbers = {}

# One publication is excluded from the study set throughout (Methods "Qualified studies and CSs").
EXCLUDED_PUBMED = ["40069456"]

## Studies, publications and ontology terms

In [ ]:
si = StudyIndex.from_parquet(session, paper.release("study")).df.cache()
gwas = si.filter(f.col("studyType") == "gwas").cache()

numbers["R1.01"] = gwas.count()
numbers["R1.02"] = gwas.filter(f.col("pubmedId").isNotNull()).select("pubmedId").distinct().count()
numbers["R1.03"] = gwas.select(f.explode("diseaseIds").alias("id")).select("id").distinct().count()
numbers["R1.04"] = len(paper.THERAPEUTIC_AREAS)
print({k: numbers[k] for k in ["R1.01", "R1.02", "R1.03", "R1.04"]})

## Ancestry composition of the study set

A study counts as having more than 10% non-European participants when non-Finnish Europeans
do not reach 90% of its LD population structure. Studies with no LD population structure are
outside this denominator.

In [ ]:
populations = (
    gwas.select("studyId", "ldPopulationStructure", "publicationDate")
    .withColumn("ldPop", f.explode("ldPopulationStructure"))
    .withColumn("year", f.year(f.to_date("publicationDate", "yyyy-MM-dd")))
    .cache()
)
predominantly_european = (f.col("ldPop.ldPopulation") == "nfe") & (f.col("ldPop.relativeSampleSize") >= 0.9)


def non_european_share(rows):
    """Percentage of studies where non-Finnish Europeans do not reach 90%."""
    total = rows.select("studyId").distinct().count()
    european = rows.filter(predominantly_european).select("studyId").distinct().count()
    return round(100 * (1 - european / total), 1)


numbers["R1.05"] = non_european_share(populations.filter(f.col("year") <= 2017))
numbers["R1.06"] = non_european_share(populations)
print({k: numbers[k] for k in ["R1.05", "R1.06"]})

## Credible sets

In [ ]:
excluded_studies = si.filter(f.col("pubmedId").isin(EXCLUDED_PUBMED)).select("studyId")
cs = (
    StudyLocus.from_parquet(session, paper.release("credible_set"))
    .df.join(excluded_studies, "studyId", "left_anti")
    .cache()
)
gwas_cs = cs.filter(f.col("studyType") == "gwas").cache()

numbers["R1.07"] = gwas_cs.count()
numbers["R1.08"] = gwas_cs.select("studyId").distinct().count()
numbers["R1.15"] = cs.filter(f.col("studyType") != "gwas").count()
numbers["R1.16"] = (
    si.join(cs.filter(f.col("studyType") != "gwas").select("studyId").distinct(), "studyId", "inner")
    .select("biosampleId")
    .distinct()
    .count()
)
print({k: numbers[k] for k in ["R1.07", "R1.08", "R1.15", "R1.16"]})

## Qualifying credible sets and the variants in them

In [ ]:
disease_cs = session.spark.read.parquet(paper.derived("qualifying_credible_sets")).select("studyLocusId")
measurement_cs = session.spark.read.parquet(paper.derived("qualifying_measurement_credible_sets")).select(
    "studyLocusId"
)

numbers["R1.10"] = disease_cs.count()
numbers["R1.11"] = measurement_cs.count()
numbers["R1.09"] = numbers["R1.10"] + numbers["R1.11"]

qualifying = cs.join(disease_cs.union(measurement_cs).distinct(), "studyLocusId", "inner").cache()
locus_variants = qualifying.select(f.explode("locus").alias("l"))

numbers["R1.13"] = qualifying.select("variantId").distinct().count()
numbers["R1.12"] = locus_variants.select("l.variantId").distinct().count()
numbers["R1.14"] = (
    locus_variants.filter(f.col("l.posteriorProbability") >= 0.9).select("l.variantId").distinct().count()
)
print({k: numbers[k] for k in ["R1.09", "R1.10", "R1.11", "R1.12", "R1.13", "R1.14"]})

## Prioritised genes, and the traits they cover

In [ ]:
diseases = session.spark.read.parquet(paper.derived("prioritised_genes_diseases")).cache()
measurements = session.spark.read.parquet(paper.derived("prioritised_genes_measurements")).cache()
target = session.spark.read.parquet(paper.release("target"))

numbers["R1.17"] = diseases.count() + measurements.count()
numbers["R1.18"] = diseases.select("geneId", f.explode("diseaseIds").alias("traitId")).distinct().count()
numbers["R1.19"] = measurements.select("geneId", f.explode("diseaseIds").alias("traitId")).distinct().count()

disease_genes = diseases.select("geneId").distinct()
measurement_genes = measurements.select("geneId").distinct()
numbers["R1.21"] = disease_genes.count()
numbers["R1.22"] = measurement_genes.count()
numbers["R1.20"] = disease_genes.union(measurement_genes).distinct().count()
numbers["R1.23"] = diseases.select(f.explode("diseaseIds").alias("traitId")).distinct().count()
numbers["R1.24"] = measurements.select(f.explode("diseaseIds").alias("traitId")).distinct().count()

protein_coding = target.filter(f.col("biotype") == "protein_coding").select("id").distinct().count()
numbers["R1.25"] = round(100 * numbers["R1.20"] / protein_coding, 1)
print({k: numbers[k] for k in ["R1.17", "R1.18", "R1.19", "R1.20", "R1.21", "R1.22", "R1.23", "R1.24", "R1.25"]})
print("protein-coding genes in the release:", protein_coding)

## Discovery over time

Cumulative discovery for the nested ancestry tiers of Figure 1c: EUR common, then non-EUR
common, then mixed common, then rare variants of any ancestry. Each tier's `layer` is what
it adds over the tier below.

In [ ]:
disease_rows = diseases.select("geneId", "diseaseIds", "year", "ancestryClass", "freqClass").toPandas()
disease_pairs = discovery.explode_pairs(disease_rows)

genes_nested = discovery.nested_tiers(disease_rows, ["geneId"], "disease genes")
pairs_nested = discovery.nested_tiers(disease_pairs, ["geneId", "traitId"], "gene-disease pairs")
fig1c = pd.concat([genes_nested, pairs_nested], ignore_index=True)
fig1c.to_csv(paper.derived("fig1c_cumulative_discovery.csv"), index=False)

fig1c.pivot_table(index="year", columns=["metric", "tier_index"], values="cumulative").tail(6)

In [ ]:
def tier_totals(nested, metric):
    """Final-year cumulative total per tier, and what each tier adds."""
    end = nested[(nested["metric"] == metric) & (nested["year"] == discovery.MAX_YEAR)].sort_values("tier_index")
    out = end[["tier", "tier_index", "cumulative"]].copy()
    out["increment"] = out["cumulative"].diff().fillna(out["cumulative"]).astype(int)
    return out.reset_index(drop=True)


gene_tiers = tier_totals(fig1c, "disease genes")
pair_tiers = tier_totals(fig1c, "gene-disease pairs")
print(gene_tiers.to_string(index=False))
print()
print(pair_tiers.to_string(index=False))

In [ ]:
# Tier 3 is all common variants of any ancestry; tier 1 is EUR common only.
for prefix, tiers in [("R1.2", gene_tiers), ("R1.3", pair_tiers)]:
    common_total = int(tiers.loc[tiers["tier_index"] == 3, "cumulative"].iloc[0])
    eur_common = int(tiers.loc[tiers["tier_index"] == 1, "cumulative"].iloc[0])
    non_eur = int(tiers.loc[tiers["tier_index"] == 2, "increment"].iloc[0])
    mixed = int(tiers.loc[tiers["tier_index"] == 3, "increment"].iloc[0])
    all_total = int(tiers.loc[tiers["tier_index"] == 4, "cumulative"].iloc[0])
    if prefix == "R1.2":
        numbers["R1.26"], numbers["R1.27"] = common_total - eur_common, common_total
        numbers["R1.28"], numbers["R1.29"] = non_eur, mixed
    else:
        numbers["R1.30"], numbers["R1.31"] = common_total - eur_common, common_total
        numbers["R1.32"], numbers["R1.33"] = non_eur, mixed
    print(
        f"{prefix}: EUR common {eur_common}, +non-EUR {non_eur}, +mixed {mixed}, all common {common_total}, incl. rare {all_total}"
    )

## Pleiotropy of disease-associated genes

The gene-level table restated: how many disease genes carry more than one disease, and more
than one therapeutic area.

In [ ]:
gene_table = session.spark.read.parquet(paper.derived("gene_table")).toPandas()
numbers["R1.37"] = int((gene_table["uniqueDiseases"] >= 2).sum())
numbers["R1.38"] = int((gene_table["uniqueTherapeuticAreas"] > 1).sum())
print({k: numbers[k] for k in ["R1.37", "R1.38"]})

## Figure 1b inputs — sample size and effect size over time

In [ ]:
for name, subset in [("qd_sl_eff", "qualifying_credible_sets"), ("qm_sl_eff", "qualifying_measurement_credible_sets")]:
    table = (
        session.spark.read.parquet(paper.derived(subset))
        .join(
            session.spark.read.parquet(paper.derived("study_annotation")).select("studyId", "year"), "studyId", "inner"
        )
        .select(
            "studyId",
            "studyLocusId",
            "year",
            f.col("studyStatistics.nSamples").alias("nSamples"),
            f.abs(f.col("rescaledStatistics.absEstimatedBeta")).alias("absBeta"),
            f.col("majorLdPopulationMaf.value").alias("maf"),
        )
        .toPandas()
    )
    table.to_csv(paper.derived(f"{name}.csv"), index=False)
    print(name, table.shape)

## Numbers

In [ ]:
print(paper.save_results("panoramic", numbers))
pd.Series(numbers).to_frame("computed")